In [ ]:
# 00_plus_auditoria · Bloco 0 — inventário das tabelas (F3, base para Porta 1)
import redivis
import pandas as pd

tabelas = [
    "df_chexpert_plus_240401",
    "DICOM_train",
    "DICOM_valid",
    "PNG_train",
    "PNG_valid",
    "CheXpert Labels",
]

for nome in tabelas:
    try:
        t = redivis.table(nome)
        t.get()
        n = t.properties.get("numRows")
        amostra = t.to_pandas_dataframe(max_results=3)
        print(f"\n=== {nome} | linhas: {n} | colunas: {amostra.shape[1]}")
        print(list(amostra.columns))
    except Exception as e:
        print(f"\n=== {nome} | ERRO: {e}")

In [ ]:
# Bloco 1 — tamanho dos pacotes (passo 3.8) e formato dos caminhos
ref = {
    "df": "df_chexpert_plus_240401:bavj",
    "dicom_train": "dicom_train:1934",
    "dicom_valid": "dicom_valid:mg73",
    "png_train": "png_train:s6cj",
    "png_valid": "png_valid:41v9",
    "labels": "chexpert_labels:pmec",
}

arquivos = {}
for k in ["dicom_train", "dicom_valid", "png_train", "png_valid", "labels"]:
    arquivos[k] = redivis.table(ref[k]).to_pandas_dataframe()
    gb = pd.to_numeric(arquivos[k]["size"]).sum() / 1e9
    print(f"{k:12s} | arquivos: {len(arquivos[k]):>7} | total: {gb:,.1f} GB")

print("\nArquivos de rótulos:")
print(arquivos["labels"][["file_name", "size"]].to_string(index=False))

df = redivis.table(ref["df"]).to_pandas_dataframe()
print("\nExemplos de caminhos:")
print("path_to_dcm    :", df["path_to_dcm"].head(2).tolist())
print("path_to_image  :", df["path_to_image"].head(2).tolist())
print("file_name DICOM:", arquivos["dicom_train"]["file_name"].head(2).tolist())

In [ ]:
# Bloco 2 — (a) quais imagens não têm DICOM; (b) formato dos arquivos de rótulo
import os

# (a) cobertura do DICOM
df["chave_dcm"] = df["path_to_dcm"].str.replace(r"^(train|valid)/", "", regex=True)
dcm = pd.concat([arquivos["dicom_train"], arquivos["dicom_valid"]])
nomes_dcm = set(dcm["file_name"])
df["tem_dicom"] = df["chave_dcm"].isin(nomes_dcm)

print("Imagens com DICOM:", df["tem_dicom"].sum(), "| sem DICOM:", (~df["tem_dicom"]).sum())
print("DICOM sem linha na tabela:", len(nomes_dcm - set(df["chave_dcm"])))
print("\nSem DICOM por vista e projeção (contagem):")
print(pd.crosstab([df["frontal_lateral"], df["ap_pa"].fillna("NA")], df["tem_dicom"]))
print("\nSem DICOM por split original:")
print(pd.crosstab(df["split"], df["tem_dicom"]))

# (b) baixar os rótulos e espiar o início de cada arquivo
os.makedirs("/scratch/labels", exist_ok=True)
for f in redivis.table(ref["labels"]).list_files():
    f.download("/scratch/labels")
for nome in sorted(os.listdir("/scratch/labels")):
    with open(f"/scratch/labels/{nome}") as fh:
        print(f"\n=== {nome} (primeiros 800 caracteres)\n", fh.read(800))

In [ ]:
# Bloco 3 — contagem de Pneumonia por arquivo de rótulo (passo 3.9, base da D-018)
rot = {}
for sec in ["report", "impression", "findings"]:
    rot[sec] = pd.read_json(f"/scratch/labels/{sec}_fixed.json", lines=True)
    v = rot[sec]["Pneumonia"]
    casam = rot[sec]["path_to_image"].isin(df["path_to_image"]).sum()
    print(f"\n=== {sec}: {len(v)} linhas | casam com a tabela principal: {casam}")
    print(v.value_counts(dropna=False).sort_index().to_string())

print("\nProtocolo: 1 = 6.042 | -1 = 18.771 | 0 = 2.806 | não mencionado = 195.843")

In [ ]:
# Bloco 4 — falta de DICOM × rótulo, idade e sexo (frontais AP/PA)
lab = rot["report"][["path_to_image", "Pneumonia"]].rename(columns={"Pneumonia": "pn_report"})
d = df.merge(lab, on="path_to_image", how="left")
d["pn_cat"] = d["pn_report"].map({1.0: "positivo", 0.0: "negativo", -1.0: "incerto"}).fillna("nao_mencionado")

fr = d[d["frontal_lateral"].eq("Frontal") & d["ap_pa"].isin(["AP", "PA"])].copy()
print("Frontais AP/PA:", len(fr))

print("\nContagem por rótulo × tem DICOM:")
print(pd.crosstab(fr["pn_cat"], fr["tem_dicom"], margins=True))

print("\nFração SEM DICOM por rótulo:")
print((1 - fr.groupby("pn_cat")["tem_dicom"].mean()).round(4).to_string())

print("\nFração SEM DICOM por projeção:")
print((1 - fr.groupby("ap_pa")["tem_dicom"].mean()).round(4).to_string())

print("\nIdade por tem DICOM:")
print(pd.to_numeric(fr["age"], errors="coerce").groupby(fr["tem_dicom"]).describe().round(1))

print("\nSexo por tem DICOM (proporção na linha):")
print(pd.crosstab(fr["sex"], fr["tem_dicom"], normalize="index").round(4))

In [ ]:
# Bloco 5 — fluxograma de exclusões e positivos esperados no teste (passo 3.9, portão G3)
d["idade"] = pd.to_numeric(d["age"], errors="coerce")
d["estudo"] = d["path_to_image"].str.extract(r"(patient\d+/study\d+)")[0]
d["n_view"] = d["path_to_image"].str.extract(r"view(\d+)_")[0].astype(int)

passos = []
x = d;                                         passos.append(("Total", x))
x = x[x["frontal_lateral"].eq("Frontal")];     passos.append(("Frontais", x))
x = x[x["ap_pa"].isin(["AP", "PA"])];          passos.append(("AP ou PA", x))
x = x[x["tem_dicom"]];                         passos.append(("Com DICOM", x))
x = x[x["idade"] >= 18];                       passos.append(("Idade >= 18 (proposta)", x))
x = (x.sort_values(["estudo", "n_view"])
       .drop_duplicates("estudo", keep="first")); passos.append(("1 frontal/estudo (proposta)", x))

for nome, t in passos:
    print(f"{nome:28s} imagens: {len(t):>7} | pacientes: {t['deid_patient_id'].nunique():>6} "
          f"| positivos: {(t['pn_cat'] == 'positivo').sum():>5}")

final = x
print("\nRótulos no conjunto final:")
print(final["pn_cat"].value_counts().to_string())

pos = (final["pn_cat"] == "positivo").sum()
print(f"\nPositivos esperados no teste (20%): ~{pos * 0.2:.0f} | meta G3: 750 (ou 500 com 2 faixas)")
print("Prevalência (incerto como negativo):", round(pos / len(final), 4))
print("Prevalência (incerto excluído):     ", round(pos / (final["pn_cat"] != "incerto").sum(), 4))
print("\nIdades < 18 nas frontais AP/PA com DICOM:",
      (d.loc[d["frontal_lateral"].eq("Frontal") & d["ap_pa"].isin(["AP", "PA"]) & d["tem_dicom"], "idade"] < 18).sum())

In [ ]:
# Bloco 6 — salvar fluxograma e elegíveis; duplicatas exatas por MD5 (passo 3.10, parte 1)
os.makedirs("/out", exist_ok=True)

flux = pd.DataFrame([{"etapa": n, "imagens": len(t),
                      "pacientes": t["deid_patient_id"].nunique(),
                      "positivos": (t["pn_cat"] == "positivo").sum()} for n, t in passos])
flux.to_csv("/out/fluxograma_plus.csv", index=False)

cols = ["path_to_image", "chave_dcm", "deid_patient_id", "estudo", "patient_report_date_order",
        "ap_pa", "idade", "sex", "pn_cat"]
final[cols].to_csv("/out/plus_elegiveis.csv", index=False)
print("Salvos: /out/fluxograma_plus.csv e /out/plus_elegiveis.csv (", len(final), "linhas )")

dcm_md5 = dcm[["file_name", "md5_hash"]].rename(columns={"file_name": "chave_dcm"})
f2 = final.merge(dcm_md5, on="chave_dcm", how="left")
dup = f2[f2.duplicated("md5_hash", keep=False)]
print("\nDICOM com MD5 repetido no conjunto final:", len(dup), "| grupos:", dup["md5_hash"].nunique())
print("Grupos com pacientes diferentes:", (dup.groupby("md5_hash")["deid_patient_id"].nunique() > 1).sum())

In [ ]:
# Bloco 7 — detalhar as duplicatas exatas
print(dup.sort_values(["md5_hash", "deid_patient_id"])
         [["md5_hash", "deid_patient_id", "estudo", "path_to_image", "ap_pa", "idade", "sex", "pn_cat"]]
         .assign(md5_hash=lambda t: t["md5_hash"].str[:8])
         .to_string(index=False))

In [ ]:
# Bloco 8 — regra de duplicatas exatas (proposta) e atualização do fluxograma
grp_pac = dup.groupby("md5_hash")["deid_patient_id"].nunique()
md5_entre = grp_pac[grp_pac > 1].index      # mesmo arquivo em pacientes diferentes
md5_mesmo = grp_pac[grp_pac == 1].index     # mesmo arquivo em estudos do mesmo paciente

f2["n_estudo"] = f2["estudo"].str.extract(r"study(\d+)")[0].astype(int)
sai_entre = f2["md5_hash"].isin(md5_entre)
mesmo = f2[f2["md5_hash"].isin(md5_mesmo)].sort_values(["md5_hash", "n_estudo"])
sai_mesmo = f2.index.isin(mesmo[mesmo.duplicated("md5_hash", keep="first")].index)

final = f2[~sai_entre & ~sai_mesmo].drop(columns=["n_estudo"])
passos.append(("Sem duplicata exata (proposta)", final))

flux = pd.DataFrame([{"etapa": n, "imagens": len(t),
                      "pacientes": t["deid_patient_id"].nunique(),
                      "positivos": (t["pn_cat"] == "positivo").sum()} for n, t in passos])
flux.to_csv("/out/fluxograma_plus.csv", index=False)
final[cols + ["md5_hash"]].to_csv("/out/plus_elegiveis.csv", index=False)

print(flux.to_string(index=False))
pos = (final["pn_cat"] == "positivo").sum()
print(f"\nPositivos esperados no teste (20%): ~{pos * 0.2:.0f}")

In [ ]:
# Bloco 9b — teste com 1 DICOM (API nova do Redivis) — passos 3.3/3.4
import os, glob, time
import pandas as pd
import redivis
import pydicom
print("pydicom", pydicom.__version__)

ref_dcm = {"train": "dicom_train:1934", "valid": "dicom_valid:mg73"}

if "final" not in globals():
    final = pd.read_csv("/out/plus_elegiveis.csv")
    print("Recarregado de /out:", len(final), "linhas")

linha = final.iloc[0]
origem = "valid" if linha["path_to_image"].startswith("valid") else "train"
os.makedirs("/scratch/dcm", exist_ok=True)

t0 = time.time()
redivis.table(ref_dcm[origem]).file(linha["chave_dcm"]).download("/scratch/dcm")
t1 = time.time()

caminho = max(glob.glob("/scratch/dcm/**/*.dcm", recursive=True), key=os.path.getmtime)
ds = pydicom.dcmread(caminho, stop_before_pixels=True)
print("arquivo:", linha["chave_dcm"], "| download:", round(t1 - t0, 1), "s |",
      round(os.path.getsize(caminho) / 1e6, 1), "MB")

campos = ["Manufacturer", "PhotometricInterpretation", "BitsStored", "BitsAllocated", "Rows", "Columns",
          "PixelSpacing", "WindowCenter", "WindowWidth", "VOILUTFunction", "PresentationLUTShape",
          "RescaleSlope", "RescaleIntercept", "RescaleType", "KVP", "XRayTubeCurrent", "ExposureTime",
          "Exposure", "ExposureInuAs", "ExposureIndex", "TargetExposureIndex", "RelativeXRayExposure",
          "ViewPosition", "BurnedInAnnotation", "ConvolutionKernel",
          "PresentationIntentType", "ImageType", "LossyImageCompression", "PixelIntensityRelationship", "SOPClassUID"]
for k in campos:
    print(f"{k:28s}", ds.get(k, "AUSENTE"))

print("\nTotal de elementos no cabeçalho:", len(ds))
os.remove(caminho)

In [ ]:
# Bloco 10 — vazão de download em paralelo (16 arquivos, 8 threads) + cabeçalhos
from concurrent.futures import ThreadPoolExecutor

amostra = pd.concat([final[final["ap_pa"] == "AP"].sample(8, random_state=2026),
                     final[final["ap_pa"] == "PA"].sample(8, random_state=2026)])

def ler(row):
    origem = "valid" if row.path_to_image.startswith("valid") else "train"
    pasta = f"/scratch/dcm/{row.Index}"
    os.makedirs(pasta, exist_ok=True)
    redivis.table(ref_dcm[origem]).file(row.chave_dcm).download(pasta)
    arq = glob.glob(f"{pasta}/**/*.dcm", recursive=True)[0]
    tam = os.path.getsize(arq)
    ds = pydicom.dcmread(arq, stop_before_pixels=True)
    reg = {"chave_dcm": row.chave_dcm, "ap_pa": row.ap_pa, "mb": tam / 1e6,
           **{k: (str(ds.get(k)) if k in ds else None) for k in campos}}
    os.remove(arq)
    return reg

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    regs = list(ex.map(ler, amostra.itertuples()))
dt = time.time() - t0

cab = pd.DataFrame(regs)
cab.to_csv("/out/cabecalhos_teste16.csv", index=False)
print(f"16 arquivos em {dt:.0f} s | {cab['mb'].sum() / dt:.1f} MB/s | {dt / 16:.1f} s por arquivo (efetivo)")
print(cab[["ap_pa", "Manufacturer", "PhotometricInterpretation", "BitsStored", "WindowCenter",
           "KVP", "XRayTubeCurrent", "Exposure", "ExposureIndex"]].to_string(index=False))

In [ ]:
# Bloco 11 — amostra de cabeçalhos (3.3/3.4): 1.000 AP + 1.000 PA, semente 2026
amostra = pd.concat([final[final["ap_pa"] == "AP"].sample(1000, random_state=2026),
                     final[final["ap_pa"] == "PA"].sample(1000, random_state=2026)])

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    regs = list(ex.map(ler, amostra.itertuples()))
print(f"{len(regs)} cabeçalhos em {(time.time() - t0) / 60:.1f} min")

cab = pd.DataFrame(regs).merge(amostra[["chave_dcm", "pn_cat"]], on="chave_dcm")
cab["fab"] = cab["Manufacturer"].fillna("AUSENTE").str.replace('"', "").str.strip().str.upper()
cab.to_csv("/out/cabecalhos_amostra2000.csv", index=False)
num = lambda s: pd.to_numeric(s, errors="coerce")

print("\nFabricante × projeção:")
print(pd.crosstab(cab["fab"], cab["ap_pa"], margins=True))

print("\nPor fabricante: fotometria, bits e janela")
print(cab.groupby("fab").agg(
    n=("fab", "size"),
    frac_mono1=("PhotometricInterpretation", lambda s: (s == "MONOCHROME1").mean()),
    frac_janela=("WindowCenter", lambda s: s.notna().mean()),
    bits=("BitsStored", lambda s: ",".join(sorted(s.dropna().unique())))).round(2).to_string())

fis = ["KVP", "XRayTubeCurrent", "ExposureTime", "Exposure", "ExposureInuAs", "ExposureIndex", "TargetExposureIndex"]
print("\nPor fabricante: fração com valor válido (> 0)")
print(cab.groupby("fab")[fis].agg(lambda s: (num(s) > 0).mean()).round(2).to_string())

print("\nPor fabricante: mediana (mín–máx) entre os válidos")
res = {}
for f in fis:
    v = num(cab[f]).where(num(cab[f]) > 0)
    g = v.groupby(cab["fab"])
    res[f] = g.median().round(1).astype(str) + " (" + g.min().round(1).astype(str) + "–" + g.max().round(1).astype(str) + ")"
print(pd.DataFrame(res).to_string())

print("\nFabricante × rótulo (proporção na linha):")
print(pd.crosstab(cab["fab"], cab["pn_cat"], normalize="index").round(3).to_string())

In [ ]:
# Bloco 12 — composição do Plus elegível (§4.3e)
final["faixa_idade"] = pd.cut(final["idade"], [18, 40, 60, 75, 200], right=False,
                              labels=["18–39", "40–59", "60–74", "75+"])
final["positivo"] = final["pn_cat"] == "positivo"
sem_inc = final[final["pn_cat"] != "incerto"]

def resumo(col):
    g = final.groupby(col, observed=True)
    out = pd.DataFrame({
        "imagens": g.size(),
        "fracao": (g.size() / len(final)).round(3),
        "prev_incerto_como_neg": g["positivo"].mean().round(4),
        "prev_sem_incerto": sem_inc.groupby(col, observed=True)["positivo"].mean().round(4),
    })
    print(f"\n=== {col}")
    print(out.to_string())

for c in ["ap_pa", "sex", "faixa_idade"]:
    resumo(c)

print("\nProjeção por faixa etária (proporção na linha):")
print(pd.crosstab(final["faixa_idade"], final["ap_pa"], normalize="index").round(3).to_string())

print("\nImagens por paciente:")
print(final.groupby("deid_patient_id").size().describe().round(1).to_string())

In [ ]:
# Bloco 13b — teste do modelo de ROI em 3 imagens, em CPU (passo 3.11)
import os, glob, time
import numpy as np, pandas as pd, redivis, pydicom, torch, cv2
import matplotlib.pyplot as plt
from lungs_segmentation.pre_trained_models import create_model

ref_dcm = {"train": "dicom_train:1934", "valid": "dicom_valid:mg73"}
final = pd.read_csv("/out/plus_elegiveis.csv")

def baixar(row, pasta="/scratch/dcm"):
    origem = "valid" if row.path_to_image.startswith("valid") else "train"
    destino = f"{pasta}/{row.Index}"
    os.makedirs(destino, exist_ok=True)
    redivis.table(ref_dcm[origem]).file(row.chave_dcm).download(destino)
    return glob.glob(f"{destino}/**/*.dcm", recursive=True)[0]

def para_8bits(ds):
    px = ds.pixel_array.astype(np.float32)
    px = (px - px.min()) / max(px.max() - px.min(), 1e-6)
    if ds.PhotometricInterpretation == "MONOCHROME1":
        px = 1.0 - px
    return (px * 255).round().astype(np.uint8)

model = create_model("resnet34").eval()

def segmentar(img8, limiar=0.2):
    # igual ao inference.py do pacote, sem GPU e sem albumentations
    x = np.repeat(img8[..., None], 3, axis=2).astype(np.float32)
    x = (x - x.min()) / max(x.max() - x.min(), 1e-6)
    x = cv2.resize(x, (512, 512), interpolation=cv2.INTER_LINEAR)
    t = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)
    with torch.no_grad():
        p = torch.sigmoid(model(t))[0].numpy()
    mascaras = (p > limiar).astype(np.uint8)       # [pulmão 1, pulmão 2]
    return x[..., 0], mascaras.max(axis=0)          # imagem 512 e ROI (união)

print("Modelo carregado | GPU:", torch.cuda.is_available())

teste = pd.concat([final[final["ap_pa"] == "AP"].sample(2, random_state=7),
                   final[final["ap_pa"] == "PA"].sample(1, random_state=7)])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, row in zip(axes, teste.itertuples()):
    arq = baixar(row)
    ds = pydicom.dcmread(arq)
    img8 = para_8bits(ds)
    t0 = time.time()
    im512, roi = segmentar(img8)
    print(f"{row.ap_pa} | original {img8.shape} | área ROI {roi.mean():.2f} | {time.time() - t0:.1f} s")
    ax.imshow(im512, cmap="gray")
    ax.contour(roi, levels=[0.5], colors="r", linewidths=1)
    ax.set_title(f"{row.ap_pa} · {row.pn_cat}")
    ax.axis("off")
    os.remove(arq)

plt.tight_layout()
plt.savefig("/out/roi_teste3.png", dpi=80)
plt.show()

In [ ]:
# Bloco 14 — ROI com mínimo-máximo puro × por percentis (mesmas 3 imagens)
def para_8bits_pct(ds, p_lo=0.5, p_hi=99.5):
    px = ds.pixel_array.astype(np.float32)
    lo, hi = np.percentile(px, [p_lo, p_hi])
    px = np.clip((px - lo) / max(hi - lo, 1e-6), 0, 1)
    if ds.PhotometricInterpretation == "MONOCHROME1":
        px = 1.0 - px
    return (px * 255).round().astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for j, row in enumerate(teste.itertuples()):
    arq = baixar(row)
    ds = pydicom.dcmread(arq)
    for i, (nome, conv) in enumerate([("mín-máx", para_8bits), ("percentis 0,5–99,5", para_8bits_pct)]):
        im512, roi = segmentar(conv(ds))
        ax = axes[i, j]
        ax.imshow(im512, cmap="gray")
        ax.contour(roi, levels=[0.5], colors="r", linewidths=1)
        ax.set_title(f"{row.ap_pa} · {nome} · área {roi.mean():.2f}")
        ax.axis("off")
    os.remove(arq)

plt.tight_layout()
plt.savefig("/out/roi_teste3_comparacao.png", dpi=80)
plt.show()

In [ ]:
# Bloco 15 — ROI final: entrada por percentis + maior componente conectado por pulmão
def maior_componente(m):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    if n <= 1:
        return m.astype(np.uint8)
    k = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (lab == k).astype(np.uint8)

def segmentar_roi(img8, limiar=0.2):
    x = np.repeat(img8[..., None], 3, axis=2).astype(np.float32)
    x = (x - x.min()) / max(x.max() - x.min(), 1e-6)
    x = cv2.resize(x, (512, 512), interpolation=cv2.INTER_LINEAR)
    t = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)
    with torch.no_grad():
        p = torch.sigmoid(model(t))[0].numpy()
    pulmoes = np.stack([maior_componente(m) for m in (p > limiar)])
    return x[..., 0], pulmoes.max(axis=0), pulmoes

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, row in zip(axes, teste.itertuples()):
    arq = baixar(row)
    ds = pydicom.dcmread(arq)
    im512, roi, pulmoes = segmentar_roi(para_8bits_pct(ds))
    areas = pulmoes.reshape(2, -1).mean(axis=1)
    ax.imshow(im512, cmap="gray")
    ax.contour(roi, levels=[0.5], colors="r", linewidths=1)
    ax.set_title(f"{row.ap_pa} · área {roi.mean():.2f} · pulmões {areas[0]:.2f}/{areas[1]:.2f}")
    ax.axis("off")
    os.remove(arq)

plt.tight_layout()
plt.savefig("/out/roi_teste3_final.png", dpi=80)
plt.show()

In [ ]:
# Bloco 16 — inspeção da ROI em 100 imagens (passo 3.11; G3: ≥ 95/100 aceitáveis)
from concurrent.futures import ThreadPoolExecutor

def baixar(row, pasta="/scratch/dcm"):
    origem = "valid" if row.path_to_image.startswith("valid") else "train"
    destino = f"{pasta}/{row.Index}"
    os.makedirs(destino, exist_ok=True)
    f = redivis.table(ref_dcm[origem]).file(row.chave_dcm)
    try:
        f.download(destino, progress=False)
    except TypeError:
        f.download(destino)
    return glob.glob(f"{destino}/**/*.dcm", recursive=True)[0]

def carregar(row):
    arq = baixar(row, pasta="/scratch/insp")
    img = para_8bits_pct(pydicom.dcmread(arq))
    os.remove(arq)
    return img

pool = final.drop(teste.index)
insp = pd.concat([pool[pool["ap_pa"] == "AP"].sample(50, random_state=2026),
                  pool[pool["ap_pa"] == "PA"].sample(50, random_state=2026)]).reset_index()
insp["n"] = range(1, 101)

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    imgs = list(ex.map(carregar, insp.itertuples()))
res = [segmentar_roi(img) for img in imgs]
del imgs
print(f"100 imagens em {time.time() - t0:.0f} s")

insp["area_roi"] = [r[1].mean() for r in res]
insp["area_p1"] = [r[2][0].mean() for r in res]
insp["area_p2"] = [r[2][1].mean() for r in res]
insp["alerta"] = ((insp["area_roi"] < 0.10) | (insp["area_roi"] > 0.45) |
                  (insp[["area_p1", "area_p2"]].min(axis=1) < 0.03))
insp[["n", "index", "chave_dcm", "ap_pa", "pn_cat", "area_roi", "area_p1", "area_p2", "alerta"]] \
    .to_csv("/out/roi_inspecao_100.csv", index=False)

for pg in range(4):
    fig, axes = plt.subplots(5, 5, figsize=(20, 20))
    for k, ax in enumerate(axes.flat):
        i = pg * 25 + k
        ax.imshow(res[i][0], cmap="gray")
        ax.contour(res[i][1], levels=[0.5], colors="r", linewidths=1)
        ax.set_title(f"#{i + 1} {insp.loc[i, 'ap_pa']}" + (" ALERTA" if insp.loc[i, "alerta"] else ""), fontsize=14)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"/out/roi_inspecao_{pg + 1}.png", dpi=60)
    plt.show()

print("Alertas automáticos:", insp.loc[insp["alerta"], "n"].tolist())
print(insp.groupby("ap_pa")["area_roi"].describe().round(3).to_string())

In [ ]:
# Bloco 17 — ampliar casos em discussão
casos = [5, 9, 23, 41, 45, 51, 56, 61, 64]
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
for ax, n in zip(axes.flat, casos):
    i = n - 1
    ax.imshow(res[i][0], cmap="gray")
    ax.contour(res[i][1], levels=[0.5], colors="r", linewidths=1.5)
    ax.set_title(f"#{n} {insp.loc[i, 'ap_pa']} · área {insp.loc[i, 'area_roi']:.2f} · "
                 f"pulmões {insp.loc[i, 'area_p1']:.2f}/{insp.loc[i, 'area_p2']:.2f}", fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.savefig("/out/roi_casos_discussao.png", dpi=70)
plt.show()

In [ ]:
# Bloco 18 — detalhes dos casos #5, #41 e #51
casos = [5, 41, 51]
sel = insp[insp["n"].isin(casos)].copy()

# rótulos do CheXbert (laudo inteiro) para outros achados
if not glob.glob("/scratch/labels/**/report_fixed.json", recursive=True):
    os.makedirs("/scratch/labels", exist_ok=True)
    for f in redivis.table("chexpert_labels:pmec").list_files():
        f.download("/scratch/labels")
rot = pd.read_json(glob.glob("/scratch/labels/**/report_fixed.json", recursive=True)[0], lines=True)
achados = ["Pneumonia", "Lung Opacity", "Consolidation", "Pleural Effusion",
           "Atelectasis", "Edema", "Support Devices", "No Finding"]
sel = sel.merge(rot[["path_to_image"] + achados], on="path_to_image", how="left")

# texto da impressão do laudo
try:
    txt = redivis.table("df_chexpert_plus_240401:bavj").to_pandas_dataframe(
        variables=["path_to_image", "section_impression"])
except TypeError:
    txt = redivis.table("df_chexpert_plus_240401:bavj").to_pandas_dataframe()[["path_to_image", "section_impression"]]
sel = sel.merge(txt, on="path_to_image", how="left")

# imagens em resolução original + cabeçalho
campos_cab = ["Manufacturer", "PhotometricInterpretation", "BitsStored", "Rows", "Columns",
              "KVP", "Exposure", "ExposureIndex"]
info = []
fig, axes = plt.subplots(3, 2, figsize=(14, 21))
for (ax1, ax2), row in zip(axes, sel.itertuples()):
    arq = baixar(row, pasta="/scratch/casos")
    ds = pydicom.dcmread(arq)
    img = para_8bits_pct(ds)
    _, roi, _ = segmentar_roi(img)
    roi_full = cv2.resize(roi, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    ax1.imshow(img, cmap="gray"); ax1.set_title(f"#{row.n} sem máscara", fontsize=13); ax1.axis("off")
    ax2.imshow(img, cmap="gray")
    ax2.contour(roi_full, levels=[0.5], colors="r", linewidths=1.5)
    ax2.set_title(f"#{row.n} {row.ap_pa} · {row.pn_cat} · ROI {row.area_roi:.2f} "
                  f"({row.area_p1:.2f}/{row.area_p2:.2f})", fontsize=13)
    ax2.axis("off")
    info.append({"n": row.n, **{k: str(ds.get(k, "AUSENTE")) for k in campos_cab}})
    os.remove(arq)
plt.tight_layout()
plt.savefig("/out/roi_casos_5_41_51.png", dpi=70)
plt.show()

# ficha de cada caso (1 = positivo, 0 = negativo, -1 = incerto, NaN = não mencionado)
tab = sel[["n", "deid_patient_id", "estudo", "ap_pa", "idade", "sex", "pn_cat"] + achados] \
        .merge(pd.DataFrame(info), on="n")
print(tab.set_index("n").T.to_string())

print("\nImpressão do laudo:")
for row in sel.itertuples():
    print(f"\n#{row.n}: {row.section_impression}")

In [ ]:
# Bloco 19 — ficha técnica das 100 inspecionadas (falhas × aceitáveis)
falhas = [5, 9, 41, 51, 56]   # ajuste aqui se o veredito final mudar
insp["falha"] = insp["n"].isin(falhas)
insp["razao_pulmoes"] = insp[["area_p1", "area_p2"]].min(axis=1) / insp[["area_p1", "area_p2"]].max(axis=1)

# rótulos: laudo inteiro (achados) + pneumonia pela impressão + texto da impressão
rot_imp = pd.read_json(glob.glob("/scratch/labels/**/impression_fixed.json", recursive=True)[0], lines=True)
ficha = (insp.merge(rot[["path_to_image"] + achados], on="path_to_image", how="left")
             .merge(rot_imp[["path_to_image", "Pneumonia"]].rename(columns={"Pneumonia": "Pneumonia_impressao"}),
                    on="path_to_image", how="left")
             .merge(txt, on="path_to_image", how="left"))

# cabeçalhos DICOM das 100
def ler_cab(row):
    arq = baixar(row, pasta="/scratch/cab100")
    ds = pydicom.dcmread(arq, stop_before_pixels=True)
    os.remove(arq)
    return {"n": row.n, **{k: (str(ds.get(k)) if k in ds else None) for k in campos_cab}}

with ThreadPoolExecutor(8) as ex:
    cabs = list(ex.map(ler_cab, ficha.itertuples()))
ficha = ficha.merge(pd.DataFrame(cabs), on="n")
ficha["fab"] = ficha["Manufacturer"].fillna("AUSENTE").str.replace('"', "").str.strip().str.upper()
ficha["familia"] = np.where(ficha["fab"].str.startswith("AGFA"), "Agfa",
                   np.where(ficha["fab"].str.startswith("GE"), "GE", "outro"))
ficha.to_csv("/out/roi_inspecao_100_ficha.csv", index=False)

# comparação falhas × aceitáveis
def tabela(col):
    t = pd.crosstab(ficha[col].astype(object).fillna("NM").astype(str), ficha["falha"])
    t = t.rename(columns={False: "aceitavel", True: "falha"})
    print(f"\n=== {col}")
    print(t.to_string())

for col in ["ap_pa", "familia", "BitsStored", "pn_cat", "Support Devices",
            "Lung Opacity", "Consolidation", "Pleural Effusion", "Atelectasis", "Edema"]:
    tabela(col)

print("\n=== Área da ROI e razão entre pulmões")
print(ficha.groupby("falha")[["area_roi", "razao_pulmoes"]].describe().round(3).T.to_string())

print("\n=== 10 menores razões entre pulmões")
print(ficha.sort_values("razao_pulmoes")[["n", "ap_pa", "familia", "pn_cat", "area_roi", "razao_pulmoes", "falha"]]
      .head(10).to_string(index=False))

print("\n=== Pneumonia: laudo inteiro (linhas) × impressão (colunas) — 1 pos, 0 neg, -1 incerto, NM não mencionado")
print(pd.crosstab(ficha["Pneumonia"].astype(object).fillna("NM"),
                  ficha["Pneumonia_impressao"].astype(object).fillna("NM")))

In [ ]:
# Bloco 20 — concordância do rótulo de pneumonia: laudo inteiro × impressão (base elegível)
fin = pd.read_csv("/out/plus_elegiveis.csv")
fin = fin[fin["sex"] != "Unknown"]          # D-029 proposta

mapa = {1.0: "positivo", 0.0: "negativo", -1.0: "incerto"}
m = (fin.merge(rot[["path_to_image", "Pneumonia"]].rename(columns={"Pneumonia": "laudo"}), on="path_to_image")
        .merge(rot_imp[["path_to_image", "Pneumonia"]].rename(columns={"Pneumonia": "impressao"}), on="path_to_image"))
m["laudo"] = m["laudo"].map(mapa).fillna("nao_mencionado")
m["impressao"] = m["impressao"].map(mapa).fillna("nao_mencionado")

print("Imagens:", len(m))
print("\nLaudo inteiro (linhas) × impressão (colunas):")
print(pd.crosstab(m["laudo"], m["impressao"], margins=True).to_string())

pl, pi = m["laudo"].eq("positivo"), m["impressao"].eq("positivo")
print(f"\nPositivos — laudo: {pl.sum()} | impressão: {pi.sum()} | nos dois: {(pl & pi).sum()} | em algum: {(pl | pi).sum()}")
print(f"Dos positivos do laudo, também positivos na impressão: {(pl & pi).sum() / pl.sum():.1%}")
print(f"Dos positivos da impressão, também positivos no laudo: {(pl & pi).sum() / pi.sum():.1%}")
print(f"Concordância exata nas 4 categorias: {(m['laudo'] == m['impressao']).mean():.1%}")
print(f"\nPositivos de teste esperados (20%) — laudo: ~{pl.sum() * 0.2:.0f} | impressão: ~{pi.sum() * 0.2:.0f} | nos dois: ~{(pl & pi).sum() * 0.2:.0f}")

In [ ]:
# Bloco 21 — os rótulos batem com o texto do laudo? (validação de alinhamento)
ref_df = "df_chexpert_plus_240401:bavj"
cols_txt = ["path_to_image", "report", "section_impression", "section_findings"]
try:
    t = redivis.table(ref_df).to_pandas_dataframe(variables=cols_txt)
except TypeError:
    t = redivis.table(ref_df).to_pandas_dataframe()[cols_txt]
fin2 = fin[["path_to_image"]].merge(t, on="path_to_image", how="left")

arquivos = {"report": ("report_fixed.json", "report"),
            "impression": ("impression_fixed.json", "section_impression"),
            "findings": ("findings_fixed.json", "section_findings")}
termos = {"Pneumonia": r"pneumonia",
          "Pleural Effusion": r"effusion",
          "Support Devices": r"\b(?:tube|catheter|pacemaker|picc|drain)\b"}

for nome, (arq, col) in arquivos.items():
    lab = pd.read_json(glob.glob(f"/scratch/labels/**/{arq}", recursive=True)[0], lines=True)
    z = fin2[["path_to_image", col]].merge(lab[["path_to_image"] + list(termos)], on="path_to_image")
    texto = z[col].fillna("").str.lower()
    print(f"\n=== arquivo {nome} × texto '{col}'")
    for achado, rx in termos.items():
        menciona = texto.str.contains(rx, regex=True)
        marcado = z[achado].notna()          # rotulador marcou 1, 0 ou -1
        print(f"{achado:17s} | texto menciona: {menciona.mean():6.1%} | "
              f"rótulo marcado quando menciona: {marcado[menciona].mean():6.1%} | "
              f"quando NÃO menciona: {marcado[~menciona].mean():6.1%}")

In [ ]:
# Bloco 22 — refazer as tabelas com o rótulo da impressão (D-018 revisada)
import os, glob
import numpy as np, pandas as pd, redivis

mapa = {1.0: "positivo", 0.0: "negativo", -1.0: "incerto"}

# rótulos da impressão
os.makedirs("/scratch/labels", exist_ok=True)
if not glob.glob("/scratch/labels/**/impression_fixed.json", recursive=True):
    for f in redivis.table("chexpert_labels:pmec").list_files():
        f.download("/scratch/labels")
imp = pd.read_json(glob.glob("/scratch/labels/**/impression_fixed.json", recursive=True)[0], lines=True)
imp = imp[["path_to_image", "Pneumonia"]].rename(columns={"Pneumonia": "pn_imp"})

# tabela principal + cobertura do DICOM
df = redivis.table("df_chexpert_plus_240401:bavj").to_pandas_dataframe()
dcm = pd.concat([redivis.table("dicom_train:1934").to_pandas_dataframe(),
                 redivis.table("dicom_valid:mg73").to_pandas_dataframe()])
df["chave_dcm"] = df["path_to_dcm"].str.replace(r"^(train|valid)/", "", regex=True)
df["tem_dicom"] = df["chave_dcm"].isin(set(dcm["file_name"]))
d = df.merge(imp, on="path_to_image", how="left")
d["pn_cat"] = d["pn_imp"].map(mapa).fillna("nao_mencionado")
d["idade"] = pd.to_numeric(d["age"], errors="coerce")
d["estudo"] = d["path_to_image"].str.extract(r"(patient\d+/study\d+)")[0]
d["n_view"] = d["path_to_image"].str.extract(r"view(\d+)_")[0].astype(int)

eleg_v1 = pd.read_csv("/out/plus_elegiveis.csv")   # resultado das regras de duplicata (v1)

# fluxograma com o rótulo da impressão
passos = []
x = d;                                         passos.append(("Total", x))
x = x[x["frontal_lateral"].eq("Frontal")];     passos.append(("Frontais", x))
x = x[x["ap_pa"].isin(["AP", "PA"])];          passos.append(("AP ou PA", x))
x = x[x["tem_dicom"]];                         passos.append(("Com DICOM", x))
x = x[x["idade"] >= 18];                       passos.append(("Idade >= 18 e registrada", x))
x = (x.sort_values(["estudo", "n_view"])
       .drop_duplicates("estudo", keep="first")); passos.append(("1 frontal por estudo", x))
x = x[x["path_to_image"].isin(eleg_v1["path_to_image"])]; passos.append(("Sem duplicata exata", x))
x = x[x["sex"].ne("Unknown")];                 passos.append(("Sexo registrado", x))
final = x.copy()

flux = pd.DataFrame([{"etapa": n, "imagens": len(t), "pacientes": t["deid_patient_id"].nunique(),
                      "positivos": (t["pn_cat"] == "positivo").sum()} for n, t in passos])
flux.to_csv("/out/fluxograma_plus_v2.csv", index=False)
print(flux.to_string(index=False))

print("\nRótulos no conjunto final (impressão):")
print(final["pn_cat"].value_counts().to_string())
pos = (final["pn_cat"] == "positivo").sum()
print(f"\nPositivos de teste esperados (20%): ~{pos * 0.2:.0f} | meta 750 (3 faixas) ou 500 (2 faixas)")
print("Prevalência (incerto como negativo):", round(pos / len(final), 4))
print("Prevalência (incerto excluído):     ", round(pos / final["pn_cat"].ne("incerto").sum(), 4))

# composição com o novo rótulo
final["faixa_idade"] = pd.cut(final["idade"], [18, 40, 60, 75, 200], right=False,
                              labels=["18–39", "40–59", "60–74", "75+"])
final["positivo"] = final["pn_cat"].eq("positivo")
sem_inc = final[final["pn_cat"].ne("incerto")]
for c in ["ap_pa", "sex", "faixa_idade"]:
    g = final.groupby(c, observed=True)
    print(f"\n=== {c}")
    print(pd.DataFrame({"imagens": g.size(),
                        "prev_incerto_como_neg": g["positivo"].mean().round(4),
                        "prev_sem_incerto": sem_inc.groupby(c, observed=True)["positivo"].mean().round(4)}).to_string())

# falta de DICOM por rótulo (frontais AP/PA)
fr = d[d["frontal_lateral"].eq("Frontal") & d["ap_pa"].isin(["AP", "PA"])]
print("\nFração SEM DICOM por rótulo (frontais AP/PA):")
print((1 - fr.groupby("pn_cat")["tem_dicom"].mean()).round(4).to_string())

# fabricante × rótulo (amostra de 2.000 cabeçalhos)
cab = pd.read_csv("/out/cabecalhos_amostra2000.csv").drop(columns=["pn_cat"], errors="ignore")
cab = cab.merge(final[["chave_dcm", "pn_cat"]], on="chave_dcm", how="inner")
print("\nFabricante × rótulo (proporção na linha, amostra de cabeçalhos):")
print(pd.crosstab(cab["fab"], cab["pn_cat"], normalize="index").round(3).to_string())

# salvar elegíveis v2
cols = ["path_to_image", "chave_dcm", "deid_patient_id", "estudo", "patient_report_date_order",
        "ap_pa", "idade", "sex", "pn_cat"]
final[cols].to_csv("/out/plus_elegiveis_v2.csv", index=False)
print("\nSalvo: /out/plus_elegiveis_v2.csv |", len(final), "linhas")